# 01 — Data Preprocessing

Runs the **preprocess** and **prepare_inputs** DVC stages to produce:
- `data/interim/articles_clean.parquet`
- `data/interim/daily_news_prices.parquet`
- `data/interim/cafef_input.parquet`

Run from the project root with the DVC remote already configured.

In [ ]:
import subprocess, yaml
from pathlib import Path

ROOT = Path().resolve()
assert (ROOT / "params.yaml").exists(), f"Run from project root, got: {ROOT}"

def run(cmd: list[str], cwd: Path = ROOT) -> None:
    subprocess.run(cmd, cwd=cwd, check=True)
    print(f"✓ {' '.join(cmd)}")

In [ ]:
# Pull raw data from DVC remote (skip if already present)
run(["dvc", "pull", "data/raw", "--run-cache", "-q"])

In [ ]:
# Run preprocessing pipeline: raw → articles_clean, daily_news_prices, cafef_input
PIPELINE = ROOT / "pipelines" / "preprocess"
run(["dvc", "repro"], cwd=PIPELINE)

In [ ]:
# Verify outputs
import pandas as pd

params  = yaml.safe_load((ROOT / "params.yaml").read_text())
interim = ROOT / params["paths"]["interim_dir"]

articles = pd.read_parquet(interim / "articles_clean.parquet")
daily    = pd.read_parquet(interim / "daily_news_prices.parquet")
cafef    = pd.read_parquet(interim / "cafef_input.parquet")

print(f"articles_clean   : {len(articles):,} rows  | cols: {list(articles.columns)}")
print(f"daily_news_prices: {len(daily):,} rows  | date range: {daily['date'].min()} → {daily['date'].max()}")
print(f"cafef_input      : {len(cafef):,} rows  | token_count p50: {cafef['token_count'].median():.0f}")

In [ ]:
# Push interim outputs so downstream Kaggle notebooks can pull them
run(["dvc", "push"])